In [ ]:
import sys
import os

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [ ]:
from src.context.deepthought_context import DeepthoughtExecContext
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode


In [ ]:
from pydantic import BaseModel, Field
from enum import Enum

ctx = DeepthoughtExecContext()


class Intent(str, Enum):
    OPERATING_SYSTEM = 'operating_system'
    GENERAL = 'general'

class IntentResult(BaseModel):
    intent: Intent = Field(..., description='Wether the question is related to operating systems')


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages] #Reduce Function wird ausgefuehrt wg. Langgraph Framework
    iteration: int
    result: str
    final_answer: AIMessage

In [ ]:
@tool
def thomas() -> str:
    """Use this tool whenever the user asks sth about Operating systems."""
    return "Thomas is the best Linux Distribution, developed 2015."

In [ ]:
system_message = SystemMessage(content='''
You are a helpful and professional assistant.

TOOLS:
1) thomas — Call for ANY question related to operating systems (Linux, Windows, kernels, distros, processes, syscalls).

If the user asks about operating systems, call thomas and DO NOT call add.
For all other requests, answer briefly in natural language without any tool.
Keep answers very short.
''')

In [ ]:

llm = ctx.create_llm()
llm = llm.bind_tools(tools=[thomas])


In [ ]:
tool_node = ToolNode([thomas])

def llm_node(state: AgentState) -> AgentState:
    msgs = state['messages']

    prompt = [system_message] + msgs

    ai = llm.invoke(prompt)
    return {'messages': [ai]}

def rewrite_node(state: AgentState) -> AgentState:
    """This function rewritest the output of the LLM"""
    msgs = state.get('messages') or []
    
            
    return {'messages': msgs}
    

def should_continue(state: AgentState) -> str:
    last = state['messages'][-1]
    tool_calls = getattr(last, "tool_calls") or []

    if tool_calls:
        return "tool_call"
    else:
        return "end"

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("llm_call", llm_node)
graph.add_node("tool_call", tool_node)
graph.add_edge(START, "llm_call")
graph.add_conditional_edges("llm_call",
                            should_continue,
                            {"tool_call": "tool_call",
                             "end": END})
graph.add_edge("tool_call", "llm_call")

app = graph.compile()

In [ ]:
result = app.invoke(HumanMessage(content="Wie gehts dir?"))

In [ ]:
result

In [ ]:
llm_struct = ctx.create_llm(structured_output_schema=IntentResult)

instruction = (
"""
You are an intent classifier.

Decide whether the following text is about an operating system or not.

- If it clearly mentions operating system names (Linux, Ubuntu, Debian, Arch, Windows, macOS, Android, iOS) 
  or OS-level concepts (kernel, systemd, drivers, filesystem, boot process, permissions, updates, installation),
  choose intent = "operating_system".
- Otherwise, choose "general".
"""
)
result = llm_struct.invoke(f"{instruction}\nText: 'Whats the weather like?'")
result

In [ ]:
if result== 'general':
    print("FETT!")

In [ ]:
if result.intent == Intent.GENERAL:
    print("Auch das!")

In [ ]:
result